# Lumis 17 (exploratory): would a few intake questions help on phone photos?

People who sent photos to SCIN also answered short questions: where on the body, what it feels like (itching,
burning, pain), what it looks like (raised, flat, flaky, fluid-filled), how long it has lasted, age group and sex.
Lumis could ask the same questions before a screening. This notebook checks whether those answers carry useful
signal for the five conditions.

**What it does**
1. Trains a small model on the answers only (no photo) and scores it on the same 482 SCIN train + validation cases
   and the same 5 folds as notebook 16. Two versions: the questions alone, and the questions plus the person's own
   guess of the kind of problem ("rash", "acne", "looks like infection").
2. Quick check on the 121 validation cases: v1's photo result combined with the answers, against v1 alone
   (combined by adding log-probabilities; nothing is tuned).
3. Lists which answers point to which condition.

**Status: exploratory.** It does not change the model and it does not use the SCIN test split. Race, ethnicity and
skin type are deliberately not used as inputs.

**Bar set before running (2026-11-13):** intake questions go into the next model comparison only if the questions
alone (without the person's own guess) reach an average sensitivity of **at least 35%** across the five conditions
(guessing gives 20%).

CPU runtime, about 2 minutes. No secrets needed. It can run in a second Colab tab while notebook 16 is running.

In [ ]:
SEED = 42
N_FOLDS = 5
C_GRID = [0.03, 0.1, 0.3, 1.0]
MAX_WEIGHT = 5.0
MIN_EXTRA_AGREEMENT = 0.25
BAR = 0.35
FLAG_PREFIXES = ('body_parts_', 'condition_symptoms_', 'other_symptoms_', 'textures_')
CATEGORICAL = ['age_group', 'sex_at_birth', 'condition_duration']
OWN_GUESS = 'related_category'

In [ ]:
!pip install -q google-cloud-storage
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'

In [ ]:
import os, io, ast, json, time, hashlib, zipfile
import numpy as np, pandas as pd
from google.cloud import storage

BUCKET = 'dx-scin-public-data'
try:
    bucket = storage.Client.create_anonymous_client().bucket(BUCKET)
    bucket.blob('dataset/scin_cases.csv').download_as_bytes(start=0, end=10)
except Exception as e:
    print('Anonymous access failed, signing in instead (a popup will ask you to allow access):', type(e).__name__)
    from google.colab import auth
    auth.authenticate_user()
    bucket = storage.Client(project='dx-scin-public').bucket(BUCKET)

def fetch(path):
    return bucket.blob(path).download_as_bytes()

cases = pd.read_csv(io.BytesIO(fetch('dataset/scin_cases.csv')), dtype={'case_id': str})
labels = pd.read_csv(io.BytesIO(fetch('dataset/scin_labels.csv')), dtype={'case_id': str})
scin = cases.merge(labels, on='case_id')
print(len(scin), 'SCIN cases')

In [ ]:
def parse(x):
    if isinstance(x, str) and x.strip():
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return {}
    return {}

def to_lumis(name):
    n = name.lower()
    if 'scabies' in n:
        return 'scabies'
    if 'psoria' in n and 'parapsoria' not in n:
        return 'psoriasis'
    if 'tinea' in n or 'dermatophyt' in n or 'ringworm' in n:
        return 'tinea'
    if n.startswith('acne') and 'rosacea' not in n and 'keloid' not in n:
        return 'acne'
    if ('eczema' in n or 'atopic' in n) and 'contact' not in n:
        return 'eczema'
    return None

def main_label(w):
    if not w:
        return None, 0.0
    name, weight = max(w.items(), key=lambda kv: kv[1])
    return to_lumis(name), weight

def fitz(r):
    vals = [r.get(f'dermatologist_fitzpatrick_skin_type_label_{i}') for i in (1, 2, 3)]
    vals = [v for v in vals if isinstance(v, str) and v.startswith('FST')]
    if not vals:
        return None
    v = int(round(np.median([int(x[3]) for x in vals])))
    return 'I-II' if v <= 2 else ('III-IV' if v <= 4 else 'V-VI')

scin['weights'] = scin.weighted_skin_condition_label.map(parse)
scin[['lumis_class', 'agreement']] = scin.weights.apply(lambda w: pd.Series(main_label(w)))
scin['tone'] = scin.apply(fitz, axis=1)

import glob
runs = pd.read_csv(f'{BASE}/exports/runs_log.csv')
V1_RUN = runs[(runs.backbone == 'efficientnetb0') & (runs.split == 'v2')].iloc[-1]['run']
cal = json.load(open(f'{BASE}/exports/runs/{V1_RUN}/calibration.json'))
CLASSES = cal['classes']
K = len(CLASSES)
IDX = {c: i for i, c in enumerate(CLASSES)}

split = pd.read_csv(f'{BASE}/data/scin_split_v1.csv', dtype={'case_id': str})
tv = split[split.split.isin(['train', 'val'])].reset_index(drop=True)          # the SCIN test split is not used
extra = scin[scin.lumis_class.notna() & (scin.agreement >= MIN_EXTRA_AGREEMENT) & (scin.agreement < 0.5)
             & ~scin.case_id.isin(set(split.case_id))][['case_id', 'lumis_class', 'agreement']].reset_index(drop=True)
print(f'{tv.case_id.nunique()} train + validation cases | {len(extra)} extra cases with weaker agreement (training only)')

## 1. The answers as model inputs

In [ ]:
def yes(col):
    return col.map(lambda v: 1.0 if (v is True or v == 1 or (isinstance(v, str) and v.strip().upper() in ('YES', 'TRUE', '1'))) else 0.0)

flag_cols = [c for c in scin.columns if c.startswith(FLAG_PREFIXES)]
cat_cols = [c for c in CATEGORICAL if c in scin.columns]
assert flag_cols, 'No question columns found in scin_cases.csv: the column names have changed.'
flags = pd.DataFrame({c: yes(scin[c]) for c in flag_cols})
cats = pd.get_dummies(scin[cat_cols].fillna('MISSING').astype(str), prefix=cat_cols).astype(float)
parts = {'questions': pd.concat([flags, cats], axis=1)}
if OWN_GUESS in scin.columns:
    guess = pd.get_dummies(scin[[OWN_GUESS]].fillna('MISSING').astype(str), prefix=[OWN_GUESS]).astype(float)
    parts['questions + own guess'] = pd.concat([flags, cats, guess], axis=1)
for k in parts:
    parts[k].index = scin.case_id.values
print(f'{len(flag_cols)} yes/no answers, {len(cat_cols)} choice questions ({cat_cols}), own guess: {OWN_GUESS in scin.columns}')
print('Yes/no answers used:', ', '.join(flag_cols))
answered = parts['questions'].loc[tv.case_id.unique(), flag_cols].sum(1) > 0
print(f'Cases with at least one yes/no answer: {answered.mean():.0%}')

## 2. Same folds as notebook 16

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore', message='The least populated class')

parent = {c: c for c in tv.case_id.unique()}
def find(c):
    while parent[c] != c:
        parent[c] = parent[parent[c]]
        c = parent[c]
    return c
for _, g in tv.groupby('md5'):
    cs = g.case_id.unique()
    for c in cs[1:]:
        parent[find(c)] = find(cs[0])
tv['group'] = tv.case_id.map(find)
ev = tv[tv.eval_photo].reset_index(drop=True)                    # one row per case
y_ev = ev.lumis_class.map(IDX).values
ev['fold'] = -1
for f, (_, te) in enumerate(StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(ev, y_ev, ev.group)):
    ev.loc[te, 'fold'] = f
print(pd.crosstab(ev.lumis_class, ev.fold, margins=True))

## 3. Answers only, cross-validated

In [ ]:
def balanced(labels, cap=None):
    labels = np.asarray(labels)
    w = len(labels) / (K * np.maximum(np.bincount(labels, minlength=K), 1))
    return (np.minimum(w, cap) if cap else w)[labels]

def macro_sens(y, p):
    return float(np.mean([(p[y == k] == k).mean() for k in range(K) if (y == k).any()]))

def scores(y, p):
    s = {'accuracy': float((p == y).mean()), 'macro_sensitivity': macro_sens(y, p)}
    for k, c in enumerate(CLASSES):
        s[f'sens_{c}'] = float((p[y == k] == k).mean()) if (y == k).any() else np.nan
    return s

y_extra = extra.lumis_class.map(IDX).values.astype(int)

def pool(X, cases, y):
    """Training set: the given cases plus the extra cases (weighted by their agreement)."""
    Xp = np.concatenate([X.loc[cases].values, X.loc[extra.case_id].values])
    yp = np.concatenate([y, y_extra])
    w = balanced(yp, MAX_WEIGHT) * np.concatenate([np.ones(len(y)), extra.agreement.values.astype(float)])
    return Xp, yp, w

def fit(X, cases, y, C):
    Xp, yp, w = pool(X, cases, y)
    return LogisticRegression(C=C, max_iter=3000).fit(Xp, yp, sample_weight=w)

def choose_C(X, cases, y):
    sc = {C: [] for C in C_GRID}
    for tr, va in StratifiedKFold(3, shuffle=True, random_state=SEED).split(cases, y):
        for C in C_GRID:
            clf = fit(X, cases[tr], y[tr], C)
            sc[C].append(macro_sens(y[va], clf.predict(X.loc[cases[va]].values)))
    return max(C_GRID, key=lambda C: (np.mean(sc[C]), -C))

rng = np.random.default_rng(SEED)
boots = [rng.integers(0, len(ev), len(ev)) for _ in range(2000)]
results, oof, chosen = {}, {}, {}
cases_ev = ev.case_id.values
for name, X in parts.items():
    pred, cs = np.zeros(len(ev), int), []
    for f in range(N_FOLDS):
        tr, te = np.where(ev.fold.values != f)[0], np.where(ev.fold.values == f)[0]
        C = choose_C(X, cases_ev[tr], y_ev[tr])
        cs.append(C)
        pred[te] = fit(X, cases_ev[tr], y_ev[tr], C).predict(X.loc[cases_ev[te]].values)
    b = [macro_sens(y_ev[i], pred[i]) for i in boots]
    results[name] = {**scores(y_ev, pred), 'macro_sens_ci95': [round(float(np.percentile(b, 2.5)), 3), round(float(np.percentile(b, 97.5)), 3)]}
    oof[name], chosen[name] = pred, cs
    print(f'{name}: C per fold {cs}')
results['always eczema (reference)'] = scores(y_ev, np.full(len(ev), IDX['eczema']))

table = pd.DataFrame(results).T
display(table[['accuracy', 'macro_sensitivity', 'macro_sens_ci95'] + [f'sens_{c}' for c in CLASSES]].round(3))
pursue = results['questions']['macro_sensitivity'] >= BAR
print(f'\nQuestions alone: average sensitivity {results["questions"]["macro_sensitivity"]:.1%} (bar {BAR:.0%}) ->',
      'worth testing with the photo model' if pursue else 'below the bar: not pursued')

## 4. Quick check: v1 photo result + answers, on the 121 validation cases

In [ ]:
fusion = None
found = sorted(glob.glob(f'{BASE}/exports/runs/*/scin_val_logits_v1.npy'))
if not found:
    print('No saved v1 validation outputs found (notebook 12 saves them): this check is skipped.')
else:
    d = os.path.dirname(found[-1])
    z1 = np.load(found[-1])
    vc = pd.read_csv(f'{d}/scin_val_cases.csv', dtype={'case_id': str})
    yv = vc.lumis_class.map(IDX).values
    v1_logp = (z1 + np.array(cal['logit_adjust'])) / cal['temperature']
    tr = ev[ev.case_id.isin(set(split[split.split == 'train'].case_id))]
    assert not set(tr.case_id) & set(vc.case_id)
    fusion = {'v1 photo only': scores(yv, v1_logp.argmax(1))}
    for name, X in parts.items():
        C = max(set(chosen[name]), key=chosen[name].count)
        clf = fit(X, tr.case_id.values, tr.lumis_class.map(IDX).values, C)       # trained without the validation cases
        logp = clf.predict_log_proba(X.loc[vc.case_id].values)
        fusion[f'{name} only'] = scores(yv, logp.argmax(1))
        fusion[f'v1 photo + {name}'] = scores(yv, (v1_logp + logp).argmax(1))
    ft = pd.DataFrame(fusion).T
    ft['correct'] = (ft.accuracy * len(yv)).round().astype(int)
    display(ft.round(3))
    print(f'{len(yv)} validation cases: {dict(zip(CLASSES, np.bincount(yv, minlength=K)))}. Small numbers: a hint, not a result.')

## 5. Which answers point to which condition

In [ ]:
X = parts['questions']
clf = fit(X, cases_ev, y_ev, max(set(chosen['questions']), key=chosen['questions'].count))
top = {}
for k, c in enumerate(CLASSES):
    order = np.argsort(-clf.coef_[k])[:6]
    top[c] = [f'{X.columns[i]} ({clf.coef_[k][i]:+.2f})' for i in order]
display(pd.DataFrame(top))

OUT = f'{BASE}/exports/intake'
os.makedirs(OUT, exist_ok=True)
json.dump({'bar': BAR, 'pursue': bool(pursue), 'cv_results': results, 'chosen_C': chosen, 'validation_fusion_check': fusion,
           'top_answers': top, 'questions_used': {'yes_no': flag_cols, 'choice': cat_cols, 'own_guess': OWN_GUESS in scin.columns},
           'n_cases': int(len(ev)), 'n_extra': int(len(extra)), 'created_at': time.strftime('%Y-%m-%d %H:%M')},
          open(f'{OUT}/exploratory_questions.json', 'w'), indent=1, default=float)
print('Saved', f'{OUT}/exploratory_questions.json')